# DIP-STER Experiments Juypter Notebook 

## Imports and initialization


In [ ]:
import torch
import numpy as np
import dipster_bare as dip
from dipster_bare.data import Sinogram, normalize
import os
from datetime import datetime
import stackview
from pathlib import Path
import mrcfile
import imageio.v2 as io
import json

%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision('high')
torch.cuda.is_available()

## Loading tiltseries

In [ ]:
# Read the data as tif
# For dipster, the rotation axis should be horizontal as seen in stackview
fname = r"Sino.tif"
data = io.volread(fname)

# Load time and angle; this code assumes a 2 columns .csv [angles, times] and will ignore the header line
t = np.loadtxt(r"AngleTime.csv", skiprows=1, delimiter=",")
angles = t[:,0]
times = t[:,1]

print(f'Loaded a {data.shape[1:]} tilt series with {data.shape[0]} projections')
stackview.slice(data)

In [ ]:
# For dipster, the rotation axis should be horizontal as seen in stackview
# If not, run this cell once
# For clarity, the series will be displayed sorted by angle, but the data to be used remain sorted by time
data = np.rot90(data, axes = (1,2))

idx = np.argsort(angles)
stackview.slice(data[idx,...])

In [ ]:
# Load the data in a sinogram helper
ts = Sinogram(data = data[:],
              angles = angles[:],
              times = times[:])

### Preprocessing

In [ ]:
# Normalize and transfer to GPU
ts = normalize(ts)
print(ts.data.max())

# /!\ Critical point: DIP is in matlab convention whereas stackview is in convention for napari
ts.data = np.transpose(ts.data, (1,2,0))

#convert numpy to tensor
ts.times = np.array(ts.times).astype('float64')
ts.times = (ts.times-np.min(ts.times))/(np.max(ts.times)-np.min(ts.times))
ts.data = torch.from_numpy(ts.data).to("cuda")
ts.angles =torch.from_numpy(ts.angles).to("cuda")
ts.times = torch.from_numpy(ts.times).to("cuda")

print(ts.data.shape)
print(ts.times.shape)
print(ts.angles.shape)

## Running the Network

### In case of crash
In the wandb folder there are auto saved networks every 1000 iterations - you can restart network from there if necessary

In [ ]:
# net = dip.Solver.from_state_dict(torch.load('your_model.pkl', map_location=torch.device('cpu')))
# net.params.noise_regularizer = 0.000005
# for key, value in net.params.to_dict().items():
#     print(key, value)
# net.eval()
# net.train(ts)

# out_name = ts_name +'_'+net.params.wandb_name
# model_path = os.path.join(directory, model_dir, out_name +'.pkl')
# torch.save(net.state_dict(), model_path)
# newnet = dip.Solver.from_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

# result = new_vmf(os.path.join(directory, out_dir, out_name +'.vmf'))
# for variables in net.reconstruct(ts=ts, slice_by_slice=True):
#     result.write_record(*variables)

### Training

The most useful parameters to update are the epochs, depth (MAPNET Depth), hidden dims (CNN depth), and noise_regularizer

In [ ]:
# Set parameters
# The most common parameters are exposed here, the full list and default values are in params.py
# Initialize the solver ------------------------------------------------------------------------------------------------------------
net = dip.Solver(ts)
vol = None

# Architecture params --------------------------------------------------------------------------------------------------------------
# MAPNET
net.params.depth = 3            # num FC layers
net.params.hidden_dim = 128     # size FC layers

# CNN
net.params.style_size= 8        # "width" of the first CNN layer <-> style^2 is the size of the latent vector
net.params.input_nch = 1        # channels, default is 1
net.params.up_factor = 16       # Controls the CNN depth, output size is style_size * up_factor


# Training params ------------------------------------------------------------------------------------------------------------------
net.params.epochs = 20                                      # Number of total epochs
net.params.training_split = 0.9                             # Ratio of data inside the training set (rest is validation)
net.params.batch_size = 4                                   # Convergence appears challenging if > 4
net.params.lr = 1e-4
net.params.gamma = 0.80                                     # Reduction factor for LR after every step size
net.params.compile_net = False                               # Minor acceleration if torch.compile is available
net.params.intval_eval = True                               # Activate intermediate validation scheme (recommended)
net.params.intval_step = 512                                # Interval between intermediate validation in steps (where total steps = epochs * batch_size * proj_size)
net.params.intval_duration = 64                             # Number of steps for the intermediate validation (nb data seen = intval_duration * batch_size)
net.params.mapnet_freeze_step = None                        # Possibility to freeze the mapnet weight after a given number of steps

# Priors
net.params.noise_regularizer = 1e-8                         # TV reg, 1e-8-1e-5 good range
net.params.output_activation = 'relu'                       # Positivity constraint

# Affine
net.params.affine_start_epoch = None                        # Epoch from which the affine is on; None => never
net.params.affine_lr = 1e-3                                 # lr for the affine param group
net.params.affine_regularizer = 1.0                         # Weight in the loss function

# Logging properties ---------------------------------------------------------------------------------------------------------------
# Saving directory
directory = r'Results\Sample1' # General folder

# Wandb logging
net.params.wandb_local_dir = directory                  # path to log, will hapen in directory \wandb
net.params.wandb_project = 'Training01'                 # Name for the Wandb project. Keep the same when testing hyperparams, results will be comparable in Wandb
net.params.eval_vol = False                             # Log an orthoslice reconstruction, will slow the training but may be useful
net.params.save_period = 1000                           # Frequency for logging results

# Initialize network -----------------------------------------------------------------------------------------------------------------

meta = net.params.to_dict()
net._setup(ts)

pytorch_total_params = sum(p.numel() for p in net.net.parameters() if p.requires_grad)
print(pytorch_total_params)

if vol is not None:
    net.eval(vol)
else:
    net.eval(save_period=net.params.save_period)

model_dir = f'{datetime.now().strftime("%Y%m%d_%H%M")}_{net.params.epochs}epochs_{net.params.lr}lr_{net.params.noise_regularizer:.2E}TV_{net.params.output_activation}Act_{net.params.wandb_name}'

model_dir = os.path.join(directory, model_dir)
Path(model_dir).mkdir(parents=True, exist_ok=True)

with open(os.path.join(model_dir, "params.json"), "w") as f:
    json.dump(net.params.to_dict(), f, indent=2, default=str)

During the training, several models are saved according to the different metrics available.
By default:
| model suffix | content |
|---|---|
|*_final.dip.pkl|State of the weight are the last step of the training (very unlikely to be the best). Useful to resume training.|
|*_train_loss.dip.pkl| Model where the train loss is minimal.|
|*_intval_loss.dip.pkl| Model where the intermediate validation loss is minimal.|
|*_intval_SSIM.dip.pkl| Model where the intermediate validation SSIM is maximal.|
|*_intval_PSNR.dip.pkl| Model where the intermediate validation PSNR is maximal.|
|*_fullval_loss.dip.pkl| Model where the full validation loss is minimal. Best model without overfitting.|
|*_fullval_SSIM.dip.pkl| Model where the full validation SSIM is maximal.|
|*_fullval_PSNR.dip.pkl| Model where the full validation PSNR is maximal.|

In [ ]:
# Run training
net.train(ts, show_summary=True, save_path=model_dir)

# Save results, model weights are saved as '.pkl' files
out_name = net.params.wandb_name        # auto coolname
model_path = os.path.join(model_dir, out_name +'_final.dip.pkl')
torch.save(net.state_dict(), model_path)

### Inference

Once a network is trained, it gains the capability to output volumes (slice-by-slice) at queried time and angle on the (t, $\theta$) manifold. 

By default, we pass the same times and angles as those of the tiltsereies, but arbitrary values can be queried.

In practice, it is often benefitial to query a single angle (e.g., 0°) and times in-between those of the projection (see the paper). This, way the interpolating nature of the network helps smoothing out the remaining artefacts (misalignments, scan distortions, contrast / brightness changes, etc.) that may not have been sufficiently corrected during preprocessing, and might be learned as wrong physical transformations

In [ ]:
# If needed, reload a previous model, otherwise the last step of training is used
# Training also saves the model with best SSIM and PSNR performance, it is typically best to load one of those
model_path = r"your_model.pkl"
net = dip.Solver.from_state_dict(torch.load(model_path, map_location=torch.device('cuda')))

In [ ]:
# By default, we export volumes at the same time and angles as the tilt series -> this will be read for the ts sinogram variable
times = ts.times.cpu().numpy()          # in [0,1]
angles = ts.angles.cpu().numpy()        # deg
suffix = ''                             # flag to facilitate naming

# Another option is to use an arbitrary manifold, typically at 0°
# Comment these lines out to keep the training time-angle coordinates
A = 0                                   # angle (°)
angles = np.zeros(ts.angles.shape) + A
suffix = f'-{A}deg-manifold_{times.shape[0]}frames'

# --------------------------------------------------------------------------------------------------------------------
# Save series of .rec files in a folder with the model name
results_dir = model_path.replace('.pkl', f'{suffix}')
Path(results_dir).mkdir(parents=True, exist_ok=True)

full_rec = []
i = 0

for vol, t in net.reconstruct(ts=ts, slice_by_slice=True, times = times, angles = angles):

    # Save vol series
    vol = np.rot90(vol.transpose(2,1,0), 1, axes = (1,2))

    with mrcfile.new(os.path.join(results_dir, f'{i}_data_{times[i]:.3f}t_{angles[i]}deg.rec'), overwrite=True) as mrc:
        mrc.set_data(vol.astype(np.float32))
        mrc.voxel_size = 1280  # Arbitrary value for training, use yours
        mrc.update_header_from_data()
    full_rec.append(vol)
    i+=1

full_rec = np.array(full_rec)
stackview.slice(full_rec, zoom_factor=2, display_min=full_rec.min(), display_max=full_rec.max())

In [ ]:
stackview.orthogonal(full_rec[0], zoom_factor=2, display_min=0, display_max = full_rec[0].max())

In [ ]:
# Export synthetic tiltseries for comparison
from dipster_bare import tomo
from dipster_bare.util import np_to_torch
from tqdm.notebook import trange

P = net.params.proj_size
dev = net.params.dev

times = ts.times      
angles = ts.angles
# angles = torch.tensor(np.ones(len(times)) * 45)

depths = torch.arange(P, device=dev)
synth = np.zeros((len(angles), P, P), dtype=np.float32)

for f in trange(len(angles)):
    ang = angles[f].reshape(1).expand(P)
    tim = times[f].reshape(1).expand(P)

    vol = net.reconstruct_slices(ang, depths, tim)             
    synth[f, :, :] = tomo.fp(vol, angles[f]).cpu().squeeze()

io.volwrite(f'{results_dir}_TiltSeries.tif', synth.astype(np.float32))

stackview.slice(synth, zoom_factor=2, display_min=0, display_max=1)

In [ ]:
io.volwrite(f'{results_dir}_TiltSeries_Raw.tif', np.transpose(ts.data.cpu().numpy().squeeze().astype(np.float32), (2, 0, 1)))

In [ ]:
I = np.argsort(angles.cpu())
stackview.slice(synth[I], zoom_factor=2, display_min=0, display_max=1)